<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# A coding-agent fleet on Agent Substrate, driven from a Workspace

This notebook runs **inside a Kubeflow Workspace** and uses it as the control point for a **fleet of coding agents**. Each agent is an [AX](https://github.com/google/ax) `Task`, and each `Task` is an [Agent Substrate](https://github.com/agent-substrate/substrate) **actor**: a gVisor sandbox that Substrate packs onto a few warm worker pods, snapshots to GCS when suspended, and restores onto any free worker when resumed.

The job is a **bug bash**. The small `textkit` library in `project/` has six bugs, each caught by one failing test module. The notebook sends one agent per bug, watches them work, suspends and resumes them, and merges the fixes back here.

```mermaid
flowchart LR
  nb["This notebook<br/>(Workspace pod)"] -->|"ax CLI"| ax["ax-server<br/>(ax-system)"]
  nb -->|"actor view"| fh["fleet-helper<br/>(ax-system)"]
  ax -->|"Control API"| api["Substrate ate-api"]
  api --> w1["worker pod 1<br/>fix-chunk, fix-roman, fix-median"]
  api --> w2["worker pod 2<br/>fix-leap-year, fix-slugify, fix-palindrome"]
  api -->|"suspend: /workspace snapshot"| gcs[("GCS")]
```

There are two agent modes:

| `AGENT_MODE` | Who fixes the bug | Needs |
| :--- | :--- | :--- |
| `scripted` (default) | A deterministic **search-based repair bot** (`project/tools/repair_bot.py`). It tries one-token mutations until the tests pass. It can fix single-token bugs but not bugs that need new code. | Nothing beyond the deploy script |
| `gemini` | An **Antigravity agent (Gemini)**, given the bug as its workspace `goal` | `GEMINI_API_KEY` set when `deploy_agent_substrate.sh` ran |

> **Prerequisite:** run `examples/agent-substrate/deploy_agent_substrate.sh` from your workstation first (see the README). The easiest way to get this notebook is to create the Workspace with the **"JupyterLab + AX agents (CPU)"** image: the `ax` CLI and `kubectl` are preinstalled and this notebook is already in `~/agent-substrate`. On any other JupyterLab image, upload this directory instead.

## Step 0: Connect to AX

The notebook finds the `ax` CLI (preinstalled in the "JupyterLab + AX" image; on other images it is downloaded from your bucket, where the deploy script put it), then checks that it can reach `ax-server` and `fleet-helper`. Both run in `ax-system`. This Workspace can reach them, but **not** the Substrate control API itself: the tenant namespace is deliberately not labeled `ate.dev/api-access=true`.

In [1]:
import json, os, subprocess, sys, time
import pandas as pd
from IPython.display import Markdown, clear_output, display

sys.path.insert(0, os.getcwd())          # this directory: axfleet.py + project/
import axfleet

AX_BIN = axfleet.ensure_ax_cli()
fleet = axfleet.Fleet()                   # atespace = this Workspace's namespace

print("ax CLI:     ", AX_BIN, "|", fleet.version().splitlines()[0])
print("ax-server:  ", fleet.server)
print("atespace:   ", fleet.atespace)
helper = fleet.actors()
print("fleet-helper:", fleet.helper_url, "| actors in atespace:", len(helper.get("actors") or []))
assert subprocess.run(["kubectl", "version", "--client"], capture_output=True).returncode == 0, \
    "kubectl is needed for `ax ssh` (it port-forwards to the Substrate router)"

ax CLI:      /usr/local/bin/ax | ax version v1alpha1 (standalone redis engine)
ax-server:   ax-server.ax-system.svc.cluster.local:8080
atespace:    kubeflow-user
fleet-helper: http://fleet-helper.ax-system.svc.cluster.local:8080 | actors in atespace: 0


## Step 1: The bug bash, locally

First, run the test suite right here in the Workspace. Six test modules, six bugs.

In [2]:
def run_suite(project_dir):
    rows = []
    for bug, (module, test) in fleet.bugs.items():
        ok = subprocess.run([sys.executable, "-B", "-m", "unittest", "-q", test],
                            cwd=project_dir, capture_output=True).returncode == 0
        rows.append({"bug": bug, "module": module, "test": test, "passes": ok})
    return pd.DataFrame(rows)

baseline = run_suite(axfleet.PROJECT_DIR)
display(baseline)
print(f"{(~baseline.passes).sum()} of {len(baseline)} test modules fail")

,bug,module,test,passes
0,chunk,textkit/chunking.py,tests.test_chunk,False
1,roman,textkit/roman.py,tests.test_roman,False
2,median,textkit/stats.py,tests.test_median,False
3,leap-year,textkit/dates.py,tests.test_leap_year,False
4,slugify,textkit/slug.py,tests.test_slugify,False
5,palindrome,textkit/palindrome.py,tests.test_palindrome,False


6 of 6 test modules fail


## Step 2: Dispatch one agent per bug

`fleet.render()` builds one AX `Workspace` holding the project files, plus one `Task` per bug. The files travel **inside the manifest**, so the agents need no Git credentials and no network access to fetch code. Every Task:

* runs `run_task.sh <bug> <mode>` as its command, which finishes by writing `/workspace/result.json`;
* binds the project at `/workspace/textkit`, inside the actor's **durable directory**, which survives suspend/resume;
* sets `debug: true`, so `ax ssh` can reach it.

In `gemini` mode each Task also gets a **workspace goal** describing its bug. AX hands the goal to an Antigravity agent before the command runs.

In [3]:
AGENT_MODE = "scripted"        # or "gemini" (needs GEMINI_API_KEY at deploy time)
BUGS = list(fleet.bugs)        # all six; trim this list for a quicker run

manifest = fleet.render(BUGS, mode=AGENT_MODE)
TASKS = [axfleet.TASK_PREFIX + b for b in BUGS]
print(f"manifest: {len(manifest):,} bytes, {len(TASKS)} tasks")
print(fleet.apply(manifest))
fleet.tasks()                  # AX creates tasks suspended: no compute used yet

manifest: 27,207 bytes, 6 tasks


workspace.ax.io/textkit created
task.ax.io/fix-chunk created
task.ax.io/fix-roman created
task.ax.io/fix-median created
task.ax.io/fix-leap-year created
task.ax.io/fix-slugify created
task.ax.io/fix-palindrome created



{'fix-palindrome': 'Suspended',
 'fix-slugify': 'Suspended',
 'fix-leap-year': 'Suspended',
 'fix-median': 'Suspended',
 'fix-roman': 'Suspended',
 'fix-chunk': 'Suspended'}

### Egress, by default: none

Substrate denies **all** actor egress until the actor has an `EgressPolicy`, and AX doesn't create one. `fleet-helper` fills that gap: it gives every new actor in this atespace a default policy. With the default deploy settings, that policy is *TLS passthrough to `generativelanguage.googleapis.com` only*. The `scripted` agents need no network at all. The `gemini` agents must not start before their policy exists, because the gateway caches a "no policy" decision for about 10 seconds.

In [4]:
t0 = time.time()
while not fleet.egress_ready(TASKS):
    if time.time() - t0 > 120:
        raise TimeoutError("fleet-helper has not created egress policies; check `kubectl -n ax-system logs deploy/fleet-helper`")
    time.sleep(3)
print(f"egress policies in place for {len(TASKS)} actors after {time.time() - t0:.0f}s")

egress policies in place for 6 actors after 3s


### Resume the fleet

Resuming is what starts an agent. Substrate restores each actor from its template's golden snapshot onto a free worker. **No new Pod is scheduled.**

In [5]:
t0 = time.time()
resumed = pd.DataFrame(fleet.resume(TASKS), columns=["task", "ok", "seconds", "message"])
display(resumed[["task", "ok", "seconds"]])
print(f"{resumed.ok.sum()}/{len(resumed)} resumed in {time.time() - t0:.1f}s wall time")

,task,ok,seconds
0,fix-chunk,True,2.6
1,fix-roman,True,2.8
2,fix-median,True,2.7
3,fix-leap-year,True,2.7
4,fix-slugify,True,2.5
5,fix-palindrome,True,2.6


6/6 resumed in 2.8s wall time


## Step 3: How many agents per pod?

`fleet-helper` shows which worker pod each actor landed on. With the default `WorkerPool` (2 workers), six agents share **two pods**. The Pod-per-sandbox approach in the `agent-sandbox` example would need six.

In [6]:
def actor_table():
    view = fleet.actors()
    df = pd.DataFrame(view.get("actors") or [])
    if df.empty:
        return df
    cols = ["name", "state", "workerPod", "node", "snapshots", "egressPolicy"]
    for col in cols:
        if col not in df.columns:
            df[col] = None
    return df[df.name.isin(TASKS)][cols]

actors = actor_table()
display(actors)
if (actors.state == "RUNNING").any():
    display(actors[actors.state == "RUNNING"].groupby("workerPod").name.apply(list).rename("agents on this pod").to_frame())

,name,state,workerPod,node,snapshots,egressPolicy
0,fix-chunk,RUNNING,ax-agents-7d8bf498cf-wrdgs,gke-kubeflow-notebooks-default-pool-b7a85a49-3s05,0,True
1,fix-leap-year,RUNNING,ax-agents-7d8bf498cf-sgvhl,gke-kubeflow-notebooks-default-pool-b7a85a49-3s05,0,True
2,fix-median,RUNNING,ax-agents-7d8bf498cf-p6h9z,gke-kubeflow-notebooks-default-pool-e835bef8-zr6w,0,True
3,fix-palindrome,RUNNING,ax-agents-7d8bf498cf-p6h9z,gke-kubeflow-notebooks-default-pool-e835bef8-zr6w,0,True
4,fix-roman,RUNNING,ax-agents-7d8bf498cf-sgvhl,gke-kubeflow-notebooks-default-pool-b7a85a49-3s05,0,True
5,fix-slugify,RUNNING,ax-agents-7d8bf498cf-fv2kw,gke-kubeflow-notebooks-default-pool-66c1d311-1rmm,0,True


,agents on this pod
workerPod,
ax-agents-7d8bf498cf-fv2kw,[fix-slugify]
ax-agents-7d8bf498cf-p6h9z,"[fix-median, fix-palindrome]"
ax-agents-7d8bf498cf-sgvhl,"[fix-leap-year, fix-roman]"
ax-agents-7d8bf498cf-wrdgs,[fix-chunk]


## Step 4: Live leaderboard

Each agent writes `/workspace/result.json` when it finishes. The notebook polls it with `ax ssh <task> -- cat /workspace/result.json`. `fixed` means the bug's tests pass **and** the agent left the tests alone; the notebook checks that itself by hashing the agent's `tests/` against its own copy.

In [7]:
def show(results):
    clear_output(wait=True)
    display(Markdown(f"**{sum(1 for r in results.values() if r)}/{len(TASKS)} agents finished** ({time.strftime('%H:%M:%S')})"))
    display(fleet.leaderboard(results))

results = fleet.wait_for_results(TASKS, timeout=1200 if AGENT_MODE == "gemini" else 300, on_poll=show)
board = fleet.leaderboard(results)

**6/6 agents finished** (02:47:59)

,task,phase,mode,fixed,tests_modified,modules_passing,patch_lines,seconds
0,fix-chunk,Running,scripted,True,False,1/6,2,8.1
1,fix-leap-year,Running,scripted,True,False,1/6,2,9.5
2,fix-median,Running,scripted,True,False,1/6,2,6.6
3,fix-palindrome,Running,scripted,True,False,1/6,3,4.5
4,fix-roman,Running,scripted,True,False,1/6,2,11.8
5,fix-slugify,Running,scripted,True,False,1/6,2,3.2


## Step 5: Look inside an agent

`ax ssh` runs commands in a live agent sandbox. Below are the winning patch for one bug and the agent's own log.

In [8]:
pick = board[board.fixed == True].task.iloc[0] if (board.fixed == True).any() else TASKS[0]
print(f"# {pick}\n")
print(fleet.patch(results[pick]) or "(no change)")
print(fleet.ssh(pick, "sh", "-c", "tail -n 5 /workspace/agent.log 2>/dev/null; ls -la /workspace").stdout)

# fix-chunk

--- a/textkit/chunking.py
+++ b/textkit/chunking.py
@@ -24,4 +24,4 @@
     if size <= 0:
         raise ValueError("size must be positive")
     items = list(items)
-    return [items[i:i + size] for i in range(0, len(items) - 1, size)]
+    return [items[i:i + size] for i in range(0, len(items) - 0, size)]



FIXED after 8 candidates: line 27: '1' -> '0'
total 8
drwxrwxrwt 3 root root  100 Oct 16 02:47 .
drwx------ 1 root root  100 Oct 16 02:47 ..
-rw-r--r-- 1 root root   46 Oct 16 02:47 agent.log
-rw-r--r-- 1 root root 7573 Oct 16 02:47 result.json
drwxr-xr-x 5 root root  140 Oct 16 02:47 textkit



## Step 6: Suspend everything, keep everything

Suspending an agent snapshots its durable `/workspace` to GCS and frees its worker. AX uses Substrate's `VOLUMES` fidelity, so **files** survive but **processes and `/tmp`** don't. The next cell proves both:

1. drop a marker in `/tmp` and note that `/workspace/result.json` exists,
2. suspend the whole fleet. The actors no longer occupy a worker pod,
3. resume one agent and time it. `result.json` is still there and the `/tmp` marker is gone.

In [9]:
probe = pick
fleet.ssh(probe, "sh", "-c", "echo transient > /tmp/marker")

t0 = time.time()
suspended = pd.DataFrame(fleet.suspend(TASKS), columns=["task", "ok", "seconds", "message"])
print(f"suspended {suspended.ok.sum()}/{len(TASKS)} agents in {time.time() - t0:.1f}s")
time.sleep(5)
display(actor_table())          # state SUSPENDED, no workerPod, snapshots >= 1

suspended 6/6 agents in 1.1s


,name,state,workerPod,node,snapshots,egressPolicy
0,fix-chunk,SUSPENDED,None,None,1,True
1,fix-leap-year,SUSPENDED,None,None,1,True
2,fix-median,SUSPENDED,None,None,1,True
3,fix-palindrome,SUSPENDED,None,None,1,True
4,fix-roman,SUSPENDED,None,None,1,True
5,fix-slugify,SUSPENDED,None,None,1,True


In [10]:
t0 = time.time()
print(fleet.resume([probe])[0][:3], f"resume took {time.time() - t0:.1f}s")
check = fleet.ssh(probe, "sh", "-c",
                  "test -f /workspace/result.json && echo 'result.json: kept'; "
                  "test -f /tmp/marker && echo '/tmp/marker: kept' || echo '/tmp/marker: gone'")
print(check.stdout)

('fix-chunk', True, 0.4) resume took 0.4s


result.json: kept
/tmp/marker: gone



## Step 7: Merge the fixes back into the Workspace

Apply every winning patch to a copy of the project **here in the Workspace** and run the full suite. In `scripted` mode the repair bot fixes the four single-token bugs (`chunk`, `roman`, `median`, `leap-year`). `slugify` and `palindrome` need new code, which is what the `gemini` agents are for.

In [11]:
import shutil

merged = os.path.expanduser("~/textkit-merged")
shutil.rmtree(merged, ignore_errors=True)
shutil.copytree(axfleet.PROJECT_DIR, merged, ignore=shutil.ignore_patterns("__pycache__"))

for _, row in board[board.fixed == True].iterrows():
    sources = results[row.task]["sources"]
    module = fleet.bugs[row.task[len(axfleet.TASK_PREFIX):]][0]
    with open(os.path.join(merged, module), "w") as f:   # take only the agent's own module
        f.write(sources[module])

after = run_suite(merged)
display(baseline[["bug", "passes"]].merge(after[["bug", "passes"]], on="bug", suffixes=("_before", "_after")))
print(f"merged into {merged}: {after.passes.sum()}/{len(after)} test modules pass")

,bug,passes_before,passes_after
0,chunk,False,True
1,roman,False,True
2,median,False,True
3,leap-year,False,True
4,slugify,False,True
5,palindrome,False,True


merged into /home/jovyan/textkit-merged: 6/6 test modules pass


## Step 8: Leave for the day (two layers of snapshots)

Suspended agents cost nothing but GCS storage, and they resume where they left off (`fleet.resume([...])`). If this Workspace uses the **resumable** kind, pausing it from the Workspaces UI also checkpoints this notebook's kernel with GKE Pod Snapshots. With both layers suspended, the agents and your notebook use no compute until you come back.

When you are done, delete the fleet. Deleting tasks removes their actors, templates and snapshots.

In [12]:
CLEANUP = False   # set to True to delete the fleet
if CLEANUP:
    display(pd.DataFrame(fleet.delete_all(), columns=["task", "ok", "seconds", "message"])[["task", "ok"]])
else:
    fleet.suspend_all()
    print("Fleet suspended (not deleted). Set CLEANUP = True and re-run to delete it.")

Fleet suspended (not deleted). Set CLEANUP = True and re-run to delete it.
